In [8]:
import sqlite3, pandas as pd, os

db_path = os.path.abspath("../data/target.db")
conn = sqlite3.connect(db_path)

runs = pd.read_sql("""
SELECT run_id, start_time, end_time, status, total_raw, total_clean, total_rejected
FROM migration_runs
ORDER BY start_time DESC
LIMIT 10;
""", conn)

conn.close()
runs


,run_id,start_time,end_time,status,total_raw,total_clean,total_rejected
0,5504560e-8eff-42f3-81f7-3c3693140962,2026-02-11T19:42:16Z,2026-02-11T19:42:16Z,SUCCESS,1050,96,954


In [9]:
import sqlite3, pandas as pd, os
from datetime import datetime, timezone

db_path = os.path.abspath("../data/target.db")
conn = sqlite3.connect(db_path)

# Use latest SUCCESS run
latest = pd.read_sql("""
SELECT run_id
FROM migration_runs
WHERE status = 'SUCCESS'
ORDER BY start_time DESC
LIMIT 1;
""", conn)

if latest.empty:
    raise ValueError("No SUCCESS runs found in migration_runs.")

RUN_ID = latest.iloc[0]["run_id"]
print("Using latest RUN_ID:", RUN_ID)

run_df = pd.read_sql("SELECT * FROM migration_runs WHERE run_id = ?", conn, params=(RUN_ID,))
if run_df.empty:
    raise ValueError("RUN_ID not found in migration_runs")

run_row = run_df.iloc[0]  # <-- this is a Series (one row)



# Critical checks
recon = pd.read_sql("""
SELECT CASE WHEN total_raw = (total_clean + total_rejected)
            THEN 'PASS' ELSE 'FAIL' END AS recon_status
FROM migration_runs WHERE run_id = ?;
""", conn, params=(RUN_ID,)).iloc[0]["recon_status"]

active_kyc_violations = pd.read_sql("""
SELECT COUNT(*) AS cnt
FROM customers_clean
WHERE run_id = ?
  AND account_status = 'ACTIVE'
  AND kyc_status <> 'VERIFIED';
""", conn, params=(RUN_ID,)).iloc[0]["cnt"]

negative_balance = pd.read_sql("""
SELECT COUNT(*) AS cnt
FROM customers_clean
WHERE run_id = ?
  AND balance < 0;
""", conn, params=(RUN_ID,)).iloc[0]["cnt"]

bad_currency = pd.read_sql("""
SELECT COUNT(*) AS cnt
FROM customers_clean
WHERE run_id = ?
  AND currency IS NOT NULL
  AND currency NOT IN ('USD','INR');
""", conn, params=(RUN_ID,)).iloc[0]["cnt"]

dup_clean = pd.read_sql("""
SELECT COUNT(*) AS cnt
FROM (
  SELECT customer_id
  FROM customers_clean
  WHERE run_id = ?
  GROUP BY customer_id
  HAVING COUNT(*) > 1
) t;
""", conn, params=(RUN_ID,)).iloc[0]["cnt"]

reject_summary = pd.read_sql("""
SELECT reject_reason, COUNT(*) AS cnt
FROM customers_rejected
WHERE run_id = ?
GROUP BY reject_reason
ORDER BY cnt DESC;
""", conn, params=(RUN_ID,))

conn.close()

# Determine pass/fail for sign-off
critical_fail = (recon != "PASS") or (dup_clean > 0) or (active_kyc_violations > 0) or (negative_balance > 0) or (bad_currency > 0)
final_status = "PASS" if not critical_fail else "FAIL"

generated_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

lines = []
lines.append("ENTERPRISE DATA MIGRATION – CRITICAL DATA QUALITY REPORT")
lines.append("=" * 62)
lines.append(f"Run ID          : {RUN_ID}")
lines.append(f"Generated (UTC) : {generated_at}")
lines.append("")
lines.append("Record Counts")
lines.append("-" * 20)
lines.append(f"Source File     : {run_row['source_file']}")
lines.append(f"Total raw records      : {int(run_row['total_raw'])}")
lines.append(f"Total clean loaded     : {int(run_row['total_clean'])}")
lines.append(f"Total rejected (audit) : {int(run_row['total_rejected'])}")
lines.append("")
lines.append("Reconciliation")
lines.append("-" * 20)
lines.append(f"Raw = Clean + Rejected : {recon}")
lines.append("")
lines.append("Critical Business Validations")
lines.append("-" * 32)
lines.append(f"Duplicate keys in clean target         : {'PASS' if dup_clean == 0 else 'FAIL'} (count={dup_clean})")
lines.append(f"ACTIVE accounts without VERIFIED KYC   : {'PASS' if active_kyc_violations == 0 else 'FAIL'} (count={active_kyc_violations})")
lines.append(f"Negative balances in target            : {'PASS' if negative_balance == 0 else 'FAIL'} (count={negative_balance})")
lines.append(f"Unsupported currency values in target  : {'PASS' if bad_currency == 0 else 'FAIL'} (count={bad_currency})")
lines.append("")
lines.append("Rejected Records Summary (Top Reasons)")
lines.append("-" * 36)

# Show top 10 reasons
top = reject_summary.head(10)
for _, r in top.iterrows():
    lines.append(f"- {r['reject_reason']}: {int(r['cnt'])}")

lines.append("")
lines.append("Final Sign-off Status")
lines.append("-" * 22)
lines.append(f"Migration Status: {final_status}")
lines.append("")

report_text = "\n".join(lines)

# Save report
os.makedirs("../reports", exist_ok=True)
report_path = f"../reports/data_quality_report_{RUN_ID}.txt"
with open(report_path, "w") as f:
    f.write(report_text)

print(report_text)
print("\nSaved:", report_path)


Using latest RUN_ID: 5504560e-8eff-42f3-81f7-3c3693140962
ENTERPRISE DATA MIGRATION – CRITICAL DATA QUALITY REPORT
Run ID          : 5504560e-8eff-42f3-81f7-3c3693140962
Generated (UTC) : 2026-02-11T19:46:38+00:00

Record Counts
--------------------
Source File     : data/legacy_customers.csv
Total raw records      : 1050
Total clean loaded     : 96
Total rejected (audit) : 954

Reconciliation
--------------------
Raw = Clean + Rejected : PASS

Critical Business Validations
--------------------------------
Duplicate keys in clean target         : PASS (count=0)
ACTIVE accounts without VERIFIED KYC   : PASS (count=0)
Negative balances in target            : PASS (count=0)
Unsupported currency values in target  : PASS (count=0)

Rejected Records Summary (Top Reasons)
------------------------------------
- invalid_signup_date: 310
- invalid_signup_date;missing_account_status: 82
- invalid_signup_date;active_requires_verified_kyc: 78
- invalid_signup_date;invalid_currency: 51
- missing_acc